In [1]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark import StorageLevel

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1785811672160_0002,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


In [2]:
spark.conf.set("spark.sql.shuffle.partitions","600")

In [3]:
ml_df=spark.read.parquet("s3://airline-dataset-2020-2025/GoldM1/ML_DATASET/")

In [4]:
ml_df.columns

['DestAirportKey', 'OriginAirportKey', 'MarketingAirlineKey', 'RouteKey', 'FlightKey', 'FlightDate', 'Quarter', 'DayofMonth', 'DayOfWeek', 'DepartureHour', 'ArrivalHour', 'DeparturePeriod', 'ArrivalPeriod', 'PeakHourIndicator', 'WeekendIndicator', 'SeasonIndicator', 'OperatingAirlineKey', 'Distance', 'ScheduledElapsedTimeMinutes', 'DistanceCategory', 'CodeshareFlag', 'IntraStateRouteFlag', 'ArrDel15', 'AirlineReliabilityScore', 'OriginAirportReliabilityScore', 'DestAirportReliabilityScore', 'RouteReliabilityScore', 'AirlineFlightCount', 'OriginAirportFlightCount', 'DestAirportFlightCount', 'RouteFlightCount', 'RouteAvgDistance', 'RouteAvgElapsedTime', 'RouteHistoricalDelayRate', 'AirlineMonthlyDelayRate', 'OriginMonthlyDelayRate', 'DestMonthlyDelayRate', 'DatasetSplit', 'ReliabilityFeatureScope', 'Year', 'Month']

In [5]:
ml_df.printSchema()

root
 |-- DestAirportKey: string (nullable = true)
 |-- OriginAirportKey: string (nullable = true)
 |-- MarketingAirlineKey: string (nullable = true)
 |-- RouteKey: string (nullable = true)
 |-- FlightKey: string (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepartureHour: integer (nullable = true)
 |-- ArrivalHour: integer (nullable = true)
 |-- DeparturePeriod: string (nullable = true)
 |-- ArrivalPeriod: string (nullable = true)
 |-- PeakHourIndicator: integer (nullable = true)
 |-- WeekendIndicator: integer (nullable = true)
 |-- SeasonIndicator: string (nullable = true)
 |-- OperatingAirlineKey: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- ScheduledElapsedTimeMinutes: integer (nullable = true)
 |-- DistanceCategory: string (nullable = true)
 |-- CodeshareFlag: integer (nullable = true)
 |-- IntraStateRouteFlag: inte

In [6]:
ml_df.count()

39895374

In [7]:
print(len(ml_df.columns))

41

In [8]:
#Split
train_df=ml_df.filter(F.col("DatasetSplit")=="Train")
validation_df=ml_df.filter(F.col("DatasetSplit")=="Validation")
test_df=ml_df.filter(F.col("DatasetSplit")=="Test")

In [9]:
categorical_cols = [

"SeasonIndicator",
"DeparturePeriod",
"ArrivalPeriod",
"DistanceCategory"

]

In [10]:
numeric_cols=[
"Year","Quarter","Month","DayofMonth","DayOfWeek","DepartureHour","ArrivalHour",
"PeakHourIndicator","WeekendIndicator","Distance","ScheduledElapsedTimeMinutes",
"CodeshareFlag","IntraStateRouteFlag","AirlineReliabilityScore",
"OriginAirportReliabilityScore","DestAirportReliabilityScore","RouteReliabilityScore",
"AirlineFlightCount","OriginAirportFlightCount","DestAirportFlightCount",
"RouteFlightCount","RouteAvgDistance","RouteAvgElapsedTime",
"RouteHistoricalDelayRate","AirlineMonthlyDelayRate",
"OriginMonthlyDelayRate","DestMonthlyDelayRate"]

In [11]:
indexers=[StringIndexer(inputCol=c,outputCol=c+"_idx",handleInvalid="keep") for c in categorical_cols]
label_indexer=StringIndexer(inputCol="ArrDel15",outputCol="label",handleInvalid="keep")
assembler=VectorAssembler(inputCols=numeric_cols+[c+"_idx" for c in categorical_cols],outputCol="features")

In [12]:
#Pipeline
feature_pipeline=Pipeline(stages=indexers+[label_indexer,assembler])

In [13]:
feature_model=feature_pipeline.fit(train_df)

In [14]:
train_features=feature_model.transform(train_df).select("features","label").repartition(300).persist(StorageLevel.DISK_ONLY)

In [15]:
validation_features = (
    feature_model
    .transform(validation_df)
    .select("features","label")
    .repartition(100)
    .persist(StorageLevel.DISK_ONLY)
)

In [16]:
test_features = (
    feature_model
    .transform(test_df)
    .select("features","label")
    .repartition(100)
    .persist(StorageLevel.DISK_ONLY)
)

In [17]:
train_features.count()

24872650

In [18]:
validation_features.count()

7425229

In [19]:
test_features.count()

7597495

In [20]:
rf=RandomForestClassifier(labelCol="label",featuresCol="features",
numTrees=60,maxDepth=10,maxBins=256,featureSubsetStrategy="sqrt",
subsamplingRate=0.8,seed=42)

In [21]:
rf_model=rf.fit(train_features)

In [22]:
val_pred=rf_model.transform(validation_features)

In [23]:
test_pred=rf_model.transform(test_features)

In [24]:
print("ROC:",BinaryClassificationEvaluator(labelCol="label").evaluate(val_pred))

('ROC:', 0.6421349043603222)

In [25]:
for m in ["accuracy","weightedPrecision","weightedRecall","f1"]:
    print(m,MulticlassClassificationEvaluator(labelCol="label",metricName=m).evaluate(val_pred))

('accuracy', 0.7938611724971715)
('weightedPrecision', 0.7356728516506498)
('weightedRecall', 0.7938611724971715)
('f1', 0.7071552990056169)

In [26]:
val_pred.groupBy("label","prediction").count().show()

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  0.0|       0.0|5876263|
|  1.0|       0.0|1512742|
|  1.0|       1.0|  18338|
|  0.0|       1.0|  17886|
+-----+----------+-------+

In [27]:
feature_names=numeric_cols+[c+"_idx" for c in categorical_cols]
for f,i in sorted(zip(feature_names,rf_model.featureImportances.toArray()),key=lambda x:x[1],reverse=True):
    print(f,round(i,6))

('Year', 0.160777)
('ArrivalPeriod_idx', 0.117635)
('DepartureHour', 0.116795)
('AirlineMonthlyDelayRate', 0.112324)
('DeparturePeriod_idx', 0.085001)
('OriginMonthlyDelayRate', 0.073606)
('RouteHistoricalDelayRate', 0.065897)
('ArrivalHour', 0.059142)
('RouteReliabilityScore', 0.057958)
('DestMonthlyDelayRate', 0.04312)
('AirlineReliabilityScore', 0.03287)
('SeasonIndicator_idx', 0.02251)
('AirlineFlightCount', 0.012992)
('OriginAirportReliabilityScore', 0.007746)
('Month', 0.006875)
('CodeshareFlag', 0.006025)
('DestAirportReliabilityScore', 0.004226)
('Quarter', 0.002871)
('DayOfWeek', 0.002306)
('PeakHourIndicator', 0.002184)
('DayofMonth', 0.002104)
('OriginAirportFlightCount', 0.001417)
('ScheduledElapsedTimeMinutes', 0.0007)
('Distance', 0.000633)
('DestAirportFlightCount', 0.000623)
('RouteFlightCount', 0.000539)
('RouteAvgElapsedTime', 0.000484)
('RouteAvgDistance', 0.000374)
('DistanceCategory_idx', 0.000154)
('IntraStateRouteFlag', 5.8e-05)
('WeekendIndicator', 5.1e-05)

In [28]:
feature_model.write().overwrite().save("s3://airline-dataset-2020-2025/GoldM1/MODEL/feature_pipeline")
rf_model.write().overwrite().save("s3://airline-dataset-2020-2025/GoldM1/MODEL/random_forest")